notebook_01_translation.ipynb

Phase 1 — Audit → Clean → Translate

Input:  产品目录_ProductCatalog_Mar_6.xlsx  (raw Chinese source)

Output: ProductCatalog_TRANSLATED_v2.xlsx  (clean, translated)

In [ ]:
!pip install openai pandas openpyxl tqdm -q

In [49]:
import os
import re
import json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY', '')

if not api_key:
    import getpass
    api_key = getpass.getpass('OpenAI API key: ')

client = OpenAI(api_key=api_key)
MODEL = 'gpt-4.1-mini'
print(f'Model: {MODEL}')

Model: gpt-4.1-mini


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload 产品目录_ProductCatalog_Mar_6.xlsx

# Get the actual file name from the uploaded dictionary
# Assuming only one file is uploaded, otherwise might need more sophisticated logic
CN_FILE = list(uploaded.keys())[0]

df_raw = pd.read_excel(CN_FILE, sheet_name='产品目录')
print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head(2)

Saving 产品目录_ProductCatalog_Mar.6.xlsx to 产品目录_ProductCatalog_Mar.6.xlsx
Shape: (114, 11)
Columns: ['产品ID', '产品名称', '英文名称', '产品类型', '中文使用说明', '使用方法(兑水)', '对应农作物/植物', '规格', '数据来源', '主要成分', '使用方法和剂量']


,产品ID,产品名称,英文名称,产品类型,中文使用说明,使用方法(兑水),对应农作物/植物,规格,数据来源,主要成分,使用方法和剂量
0,AF0001,柑橘菌立清,Citrus Bacteria Clear,辅助生长,核心防治对象（柑橘全生育期病害）\n1. 真菌病害（核心）：根腐病、霜霉病、灰霉病、紫斑病、...,1克兑水1斤,登记作物：辣椒。实践表明在柑橘、蔬菜、果树、瓜果、大田、中药材、花卉、苗木、茶树等多种作物上...,50克/500克,Excel原表+图片标签,有效菌种：枯草芽孢杆菌、侧孢短芽孢杆菌、地衣芽孢杆菌、胶冻样芽孢杆菌、解淀粉芽孢杆菌；有效活...,作物生长期均匀稀释1000倍液，叶面均匀喷雾。可根据当地作物生长情况，用肥习惯，酌情调整用量...
1,AF0002,葱姜蒜菌立清,Onion-Ginger-Garlic Bacteria Clear,辅助生长,核心防治对象（葱姜蒜全生育期病害）\n1. 真菌病害（核心）：根腐病、霜霉病、灰霉病、紫斑病...,1克兑水1斤,登记作物：辣椒。实践表明在大葱、生姜、大蒜、蔬菜、果树、瓜果、大田、中药材、花卉、苗木、茶树...,50克/500克,Excel原表+图片标签,有效菌种：枯草芽孢杆菌、侧孢短芽孢杆菌、地衣芽孢杆菌、胶冻样芽孢杆菌、解淀粉芽孢杆菌,作物生长期均匀稀释1000倍液，叶面均匀喷雾。可根据当地作物生长情况，用肥习惯，酌情调整用量...


In [ ]:
# SECTION 1 — AUDIT  (no API calls, completely free)

print('=== BLANK CELLS PER COLUMN ===')
blank_counts = df_raw.isnull().sum()
total = len(df_raw)
for col, count in blank_counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 5)
    print(f'  {col:<25} {count:>3} / {total}  ({pct:4.1f}%)  {bar}')

=== BLANK CELLS PER COLUMN ===
  产品ID                        0 / 114  ( 0.0%)  
  产品名称                        0 / 114  ( 0.0%)  
  英文名称                        0 / 114  ( 0.0%)  
  产品类型                        0 / 114  ( 0.0%)  
  中文使用说明                      0 / 114  ( 0.0%)  
  使用方法(兑水)                    0 / 114  ( 0.0%)  
  对应农作物/植物                    0 / 114  ( 0.0%)  
  规格                          0 / 114  ( 0.0%)  
  数据来源                        0 / 114  ( 0.0%)  
  主要成分                        6 / 114  ( 5.3%)  █
  使用方法和剂量                    17 / 114  (14.9%)  ██


In [ ]:
# Identify column names — adjust if your sheet uses different headers
ID_COL          = '产品ID'
NAME_COL        = '产品名称'
TYPE_COL        = '产品类型'
CROP_COL        = '对应农作物/植物'
INGREDIENT_COL  = '主要成分'
DOSAGE_COL      = '使用方法和剂量'
PROSE_COL       = '中文使用说明'
DILUTION_COL    = '使用方法(兑水)'
SOURCE_COL      = '数据来源'

CRITICAL_COLS = [INGREDIENT_COL, DOSAGE_COL, CROP_COL]

print('=== BLANK CRITICAL FIELDS BY PRODUCT ===')
for col in CRITICAL_COLS:
    blank_ids = df_raw[df_raw[col].isnull()][ID_COL].tolist()
    if blank_ids:
        print(f'\n  {col} — {len(blank_ids)} blank:')
        for pid in blank_ids:
            print(f'    {pid}')
    else:
        print(f'\n  {col} — no blanks ✓')

=== BLANK CRITICAL FIELDS BY PRODUCT ===

  主要成分 — 6 blank:
    AF0014
    AF0017
    AF0026
    AF0029
    AF0030
    AF0035

  使用方法和剂量 — 17 blank:
    AF0004
    AF0014
    AF0015
    AF0017
    AF0024
    AF0026
    AF0027
    AF0028
    AF0029
    AF0030
    AF0031
    AF0035
    AF0036
    PN0002
    PN0026
    PN0045
    PN0047

  对应农作物/植物 — no blanks ✓


In [ ]:
# Structured vs free-text column detection
print('=== COLUMN CARDINALITY (low = structured, high = free-text) ===')
for col in df_raw.columns:
    n_unique = df_raw[col].dropna().nunique()
    avg_len  = df_raw[col].dropna().astype(str).str.len().mean()
    kind     = 'free-text prose' if avg_len > 80 else ('categorical' if n_unique < 20 else 'semi-structured')
    print(f'  {col:<25} unique={n_unique:>3}  avg_len={avg_len:>6.1f}  → {kind}')

=== COLUMN CARDINALITY (low = structured, high = free-text) ===
  产品ID                      unique=114  avg_len=   6.0  → semi-structured
  产品名称                      unique=114  avg_len=   6.1  → semi-structured
  英文名称                      unique=113  avg_len=  25.8  → semi-structured
  产品类型                      unique=  2  avg_len=   3.1  → categorical
  中文使用说明                    unique=110  avg_len= 762.0  → free-text prose
  使用方法(兑水)                  unique= 36  avg_len=   9.0  → semi-structured
  对应农作物/植物                  unique= 80  avg_len=  39.3  → semi-structured
  规格                        unique= 35  avg_len=   7.1  → semi-structured
  数据来源                      unique=  6  avg_len=  11.6  → categorical
  主要成分                      unique= 69  avg_len=  23.2  → semi-structured
  使用方法和剂量                   unique= 62  avg_len= 140.1  → free-text prose


In [ ]:
# Duplicate product ID check
dupes = df_raw[df_raw.duplicated(subset=[ID_COL], keep=False)]
if len(dupes):
    print(f'WARNING: {len(dupes)} duplicate product IDs found:')
    print(dupes[[ID_COL, NAME_COL]])
else:
    print(f'No duplicate product IDs ✓  ({len(df_raw)} unique products)')

No duplicate product IDs ✓  (114 unique products)


In [ ]:
# For products where INGREDIENT_COL is blank, check if prose column has content
print('=== INGREDIENT RECOVERY POTENTIAL ===')
blank_ing = df_raw[df_raw[INGREDIENT_COL].isnull()]
for _, row in blank_ing.iterrows():
    prose = str(row.get(PROSE_COL, ''))
    has_content = len(prose.strip()) > 20
    print(f"  {row[ID_COL]} | {row[NAME_COL][:40]}")
    print(f"    prose available: {has_content} | prose excerpt: {prose[:100]}")
    print()

=== INGREDIENT RECOVERY POTENTIAL ===
  AF0014 | 白菜甘蓝水溶肥
    prose available: True | prose excerpt: 核心功效
1.促根壮苗：快速生根，根系发达，提高养分吸收效率，减少僵苗、弱苗现象，叶片浓绿肥厚，叶绿素含量提升，光合作用增强；
2.提抗逆：增强抗低温、抗干旱能力，降低苗期病害（猝倒、立枯）发生率；


  AF0017 | 一喷绿
    prose available: True | prose excerpt: 核心功效
1.快速复绿：养分可快速穿透叶片角质层，解决缺素 / 药害 / 低温导致的黄叶、小叶、卷叶、薄叶，使作物小叶变大、卷叶舒展、病叶转健转绿；
2.强化光合，提质增产：提升叶绿素含量与光合效率，

  AF0026 | 茄子1号
    prose available: True | prose excerpt: 核心功效
1.壮根稳苗，增强抗逆：强化根系细胞壁，促侧根萌发，减少烂根死苗；保障叶绿素合成，叶片浓绿厚实，提升光合效率；整体增强植株抗寒、抗旱、抗病害能力；
2.促花保果，防畸防裂：补充花期、坐果期关

  AF0029 | 绿叶先锋
    prose available: True | prose excerpt: 核心功效
1.促绿壮叶：快速修复生理性黄叶、小叶、卷叶、落叶，增厚叶片、提升叶绿素含量，增强光合作用；
2.生根抗逆：促进白根萌发、根系发达，增强抗旱、抗寒、抗药害 / 冻害能力，减少重茬与早衰；
3

  AF0030 | 催花座果
    prose available: True | prose excerpt: 核心功效
1.促花，优化花芽：提前启动花芽分化，增加雌花比例与有效花数量；让花苞饱满、花期整齐，减少畸形花、败育花；对低温、寡照导致的花芽发育不良有明显改善；
2.保花保果，提升坐果率：增强花粉活力与

  AF0035 | 百菌灵
    prose available: True | prose excerpt: 核心功效
1.防护治疗：能高效防治白粉病、黑斑病、炭疽病、锈病、叶斑病、纹枯病等真菌病害，同时对细菌性流胶病、溃疡病、穿孔病等也有抑制作用。
2.营养补充，促株健壮：可促进叶片生长，让叶片变大、

In [ ]:
# Verify the 6 products prose content before spending API calls
for pid in ['AF0014', 'AF0017', 'AF0026', 'AF0029', 'AF0030', 'AF0035']:
    row = df_raw[df_raw[ID_COL] == pid].iloc[0]
    prose = str(row[PROSE_COL])
    print(f'{pid} | {row[NAME_COL]}')
    print(f'  prose length: {len(prose)} chars')
    print(f'  first 150: {prose[:150]}')
    print()

AF0014 | 白菜甘蓝水溶肥
  prose length: 146 chars
  first 150: 核心功效
1.促根壮苗：快速生根，根系发达，提高养分吸收效率，减少僵苗、弱苗现象，叶片浓绿肥厚，叶绿素含量提升，光合作用增强；
2.提抗逆：增强抗低温、抗干旱能力，降低苗期病害（猝倒、立枯）发生率；
3.提质增产：加速心叶分化与包心，球茎紧实、均匀，减少空心、裂球现象。
防治作物：白菜、甘蓝

AF0017 | 一喷绿
  prose length: 335 chars
  first 150: 核心功效
1.快速复绿：养分可快速穿透叶片角质层，解决缺素 / 药害 / 低温导致的黄叶、小叶、卷叶、薄叶，使作物小叶变大、卷叶舒展、病叶转健转绿；
2.强化光合，提质增产：提升叶绿素含量与光合效率，膨果均匀、着色鲜亮、硬度提升，减少畸形果、裂果、生理落果现象；
3.促根壮苗，增强抗逆能力：激活根系

AF0026 | 茄子1号
  prose length: 254 chars
  first 150: 核心功效
1.壮根稳苗，增强抗逆：强化根系细胞壁，促侧根萌发，减少烂根死苗；保障叶绿素合成，叶片浓绿厚实，提升光合效率；整体增强植株抗寒、抗旱、抗病害能力；
2.促花保果，防畸防裂：补充花期、坐果期关键中微量营养，延长授粉时间，提高座果率，减少落花落果；均衡果皮与果肉生长速度，有效预防茄子常见的畸形

AF0029 | 绿叶先锋
  prose length: 174 chars
  first 150: 核心功效
1.促绿壮叶：快速修复生理性黄叶、小叶、卷叶、落叶，增厚叶片、提升叶绿素含量，增强光合作用；
2.生根抗逆：促进白根萌发、根系发达，增强抗旱、抗寒、抗药害 / 冻害能力，减少重茬与早衰；
3.提质增产：保花保果、促进膨果着色，改善果实风味。
防治作物：大田作物（小麦、玉米、水稻等）果树类（

AF0030 | 催花座果
  prose length: 308 chars
  first 150: 核心功效
1.促花，优化花芽：提前启动花芽分化，增加雌花比例与有效花数量；让花苞饱满、花期整齐，减少畸形花、败育花；对低温、寡照导致的花芽发育不良有明显改善；
2.保花保果，提升坐果率：增强花粉活力与花粉管伸长，提高自然授粉、人工授粉成功率；抑制脱落酸合成，

In [ ]:
df_clean = df_raw.copy()

for pid in ['AF0014', 'AF0017', 'AF0026', 'AF0029', 'AF0030', 'AF0035']:
    idx = df_clean[df_clean[ID_COL] == pid].index[0]
    df_clean.loc[idx, 'ingredient_source'] = 'not_in_source'
    # Leave 主要成分 as NaN — honest null is better than hallucinated data

print('Marked 6 products as ingredient_source=not_in_source')
print('主要成分 left as NaN — no fabrication')
print(df_clean[df_clean['ingredient_source'] == 'not_in_source'][[ID_COL, NAME_COL, 'ingredient_source']])

Marked 6 products as ingredient_source=not_in_source
主要成分 left as NaN — no fabrication
      产品ID     产品名称 ingredient_source
13  AF0014  白菜甘蓝水溶肥     not_in_source
16  AF0017      一喷绿     not_in_source
25  AF0026     茄子1号     not_in_source
28  AF0029     绿叶先锋     not_in_source
29  AF0030     催花座果     not_in_source
33  AF0035      百菌灵     not_in_source


In [ ]:
# Save audit report
audit = {
    'total_products': len(df_raw),
    'columns': df_raw.columns.tolist(),
    'blank_counts': blank_counts.to_dict(),
    'blank_critical': {
        col: df_raw[df_raw[col].isnull()][ID_COL].tolist()
        for col in CRITICAL_COLS
    },
    'duplicate_ids': dupes[ID_COL].tolist() if len(dupes) else [],
    'ingredient_recovery': {
        'attempted': ['AF0014', 'AF0017', 'AF0026', 'AF0029', 'AF0030', 'AF0035'],
        'recovered': [],
        'unrecoverable': ['AF0014', 'AF0017', 'AF0026', 'AF0029', 'AF0030', 'AF0035'],
        'reason': 'Prose contains marketing/effects copy only — no chemical ingredient data present in source'
    }
}
with open('audit_report.json', 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
print('Saved → audit_report.json')
print('\nAUDIT COMPLETE — proceeding to Section 2: Clean')

Saved → audit_report.json

AUDIT COMPLETE — proceeding to Section 2: Clean


SECTION 2 — CLEAN  (no API calls, completely free)

In [ ]:
# CELL 10 — Ingredient recovery
# Recovery attempted but prose contains marketing copy only — no chemical
# ingredient data present in source for 6 products.
# These are marked ingredient_source='not_in_source' and left as NaN.
# See audit_report.json → ingredient_recovery for details.

In [ ]:
# ── CELL 11 — Flag products needing manual review
# Products with 3+ blank critical fields after recovery = manually review
df_clean['blank_critical_count'] = df_clean[CRITICAL_COLS].isnull().sum(axis=1)
needs_review = df_clean[df_clean['blank_critical_count'] >= 2][[ID_COL, NAME_COL, 'blank_critical_count']]

if len(needs_review):
    print(f'Products needing manual review ({len(needs_review)}):')
    print(needs_review.to_string())
else:
    print('No products need manual review ✓')


Products needing manual review (6):
      产品ID     产品名称  blank_critical_count
13  AF0014  白菜甘蓝水溶肥                     2
16  AF0017      一喷绿                     2
25  AF0026     茄子1号                     2
28  AF0029     绿叶先锋                     2
29  AF0030     催花座果                     2
33  AF0035      百菌灵                     2


In [ ]:
# ── CELL 12 — Save cleaned Chinese file ──────────────────────────────────────
# Drop the helper count column before saving — it was just for flagging
df_clean.drop(columns=['blank_critical_count'], inplace=True)

df_clean.to_excel('产品目录_CLEANED.xlsx', index=False)
print(f'Saved → 产品目录_CLEANED.xlsx')
print(f'Shape: {df_clean.shape}')
print(f'\nColumns:')
for col in df_clean.columns:
    print(f'  {col}')
print(f'\nSECTION 2 COMPLETE ✓')
print('Next: Section 3 — Translation')
print(f'\nSECTION 2 COMPLETE — review 产品目录_CLEANED.xlsx before proceeding')

Saved → 产品目录_CLEANED.xlsx
Shape: (114, 12)

Columns:
  产品ID
  产品名称
  英文名称
  产品类型
  中文使用说明
  使用方法(兑水)
  对应农作物/植物
  规格
  数据来源
  主要成分
  使用方法和剂量
  ingredient_source

SECTION 2 COMPLETE ✓
Next: Section 3 — Translation

SECTION 2 COMPLETE — review 产品目录_CLEANED.xlsx before proceeding


SECTION 3 — TRANSLATE  (API calls — costs money, run once)

In [ ]:
# ── CELL 13 — Helper functions ────────────────────────────────────────────────
def is_blank(value):
    """Return True if value is None, NaN, or empty/whitespace string."""
    if pd.isna(value):
        return True
    return str(value).strip() == ''

def contains_chinese(text):
    """Return True if text contains any Chinese characters."""
    if pd.isna(text):
        return False
    return bool(re.search(r'[\u4e00-\u9fff]', str(text)))

def translate_text(text, column_hint=''):
    """
    Translate Chinese text to English.
    column_hint tells the model which field this is so it applies
    the right rules for that field type.
    """
    if is_blank(text):
        return text
    if not contains_chinese(text):
        return text  # already English — skip API call

    system = """You are an agricultural product data translator.
Translate Chinese to English. Apply these field-specific rules:

- Main Ingredients (主要成分):
  Preserve exact Bacillus species names, CFU counts (e.g. 2×10⁸),
  and concentration percentages EXACTLY. Never paraphrase or round.

- Product Type (产品类型):
  Use standard English agricultural terminology only.

- Target Crops (对应农作物/植物):
  Translate crop names accurately. Keep any parenthetical examples.

- Dosage (使用方法和剂量):
  Preserve ALL numeric ratios exactly (e.g. 1:500, 1:1000).
  Never round, approximate, or paraphrase numbers.

- Usage Instructions (中文使用说明):
  Translate the full text completely.
  When you encounter disease/pest/symptom terms in the text,
  prefix each one inline with its category tag:
    [DISEASE] for fungal, bacterial, viral plant diseases
    [PEST]    for insects, mites, nematodes, molluscs, weeds
    [SYMPTOM] for physiological symptoms (yellowing, dwarfing,
              mosaic, wilting, tip-drying, curling, cracking)
  Example: "防治白粉病、蚜虫" → "Controls [DISEASE] Powdery Mildew, [PEST] Aphids"

- Dilution Method (使用方法(兑水)):
  Preserve numeric ratios exactly. Translate units accurately.

- Safety / Precautions (注意事项):
  Translate completely. Never summarize or omit any warning.

- If a cell is blank or contains only numbers/symbols, return it unchanged.
- Return ONLY the translated text, no explanations, no preamble."""

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': f'Field: {column_hint}\n\n{str(text)}'}
            ],
            temperature=0,
            max_tokens=4096,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f'  Translation error: {e}')
        return text  # return original on failure

In [ ]:
# ── CELL 14 — Translate short structured columns ─────────────────────────────
df = pd.read_excel('产品目录_CLEANED.xlsx')
print(f'Loaded cleaned file: {df.shape}')

SHORT_COLUMNS = {
    '产品名称':       'Product Name',
    '产品类型':       'Product Type',
    '对应农作物/植物': 'Target Crops',
    '规格':          'Specification',
    '主要成分':       'Main Ingredients',
    '使用方法和剂量':  'Usage Method and Dosage',
}

for cn_col, hint in SHORT_COLUMNS.items():
    if cn_col not in df.columns:
        print(f'  SKIP {cn_col} — column not found')
        continue
    print(f'Translating: {cn_col} → {hint}')
    tqdm.pandas(desc=f'  {hint}')
    df[cn_col] = df[cn_col].progress_apply(
        lambda x, h=hint: translate_text(x, column_hint=h)  # h=hint fixes closure bug
    )

print('\nShort columns done.')

Loaded cleaned file: (114, 12)
Translating: 产品名称 → Product Name


  Product Name: 100%|██████████| 114/114 [01:36<00:00,  1.19it/s]


Translating: 产品类型 → Product Type


  Product Type: 100%|██████████| 114/114 [01:30<00:00,  1.26it/s]


Translating: 对应农作物/植物 → Target Crops


...

Translating: 主要成分 → Main Ingredients


  Main Ingredients: 100%|██████████| 114/114 [01:28<00:00,  1.29it/s]


Translating: 使用方法和剂量 → Usage Method and Dosage


  Usage Method and Dosage: 100%|██████████| 114/114 [03:11<00:00,  1.68s/it]


Short columns done.


In [ ]:
# Quick quality check before checkpoint
import random

print('=== SPOT CHECK — 3 random products ===\n')
for idx in random.sample(range(len(df)), 3):
    row = df.iloc[idx]
    print(f"ID:          {row['产品ID']}")
    print(f"Name:        {row['产品名称']}")
    print(f"Type:        {row['产品类型']}")
    print(f"Crops:       {row['对应农作物/植物'][:80]}")
    print(f"Ingredients: {row['主要成分']}")
    print(f"Dosage:      {str(row['使用方法和剂量'])[:100]}")
    print()

# Check the 6 blank ingredient products — make sure they stayed NaN
print('=== BLANK INGREDIENT PRODUCTS — should still be NaN ===')
flagged = df[df['ingredient_source'] == 'not_in_source'][['产品ID', '产品名称', '主要成分']]
print(flagged.to_string())

# Check no Chinese remains in short columns
print('\n=== CHINESE REMAINING IN SHORT COLUMNS ===')
for col in ['产品名称', '产品类型', '对应农作物/植物', '主要成分', '使用方法和剂量']:
    cn_count = df[col].apply(
        lambda x: bool(re.search(r'[\u4e00-\u9fff]', str(x))) if pd.notna(x) else False
    ).sum()
    status = 'WARNING' if cn_count > 0 else 'OK'
    print(f'  {status}  {col}: {cn_count} rows still contain Chinese')

=== SPOT CHECK — 3 random products ===

ID:          PN0005
Name:        29% Insecticidal Duo
Type:        Pesticide
Crops:       Rice, Cruciferous vegetables (cabbage, Chinese cabbage, etc.), Sugarcane, Tea, C
Ingredients: Active Ingredient: Emamectin Benzoate 29%
Dosage:      | Crop/Location | Target Pest | Dosage (Formulation Amount/mu) | Application Method |
| --- | --- | 

ID:          AF0059
Name:        Macronutrient One-Spray Green
Type:        Growth Promoter
Crops:       Fruit trees, melons and fruits, vegetables such as tomatoes and peppers, cotton,
Ingredients: Product Type: Macronutrient Water-Soluble Fertilizer
Dosage:      Dilute this product with water at 800-1000 times.
Melons: Dilute 500 times, can spray 2-3 times cons

ID:          AF0009
Name:        Bags of Junliqing
Type:        Growth Promoter
Crops:       Registered crops: Chili pepper. Practice has shown significant effects on variou
Ingredients: Effective viable bacteria count ≥ 2×10⁸ CFU/ml
Dosage:      Durin

In [ ]:
# ── CELL 15 — Checkpoint save
# Save before the expensive long-column translation
df.to_excel('checkpoint_before_long_translation.xlsx', index=False)
print('Saved checkpoint → checkpoint_before_long_translation.xlsx')
print(f'Shape: {df.shape}')


Saved checkpoint → checkpoint_before_long_translation.xlsx
Shape: (114, 12)


In [ ]:
# ── CELL 16 — Translate long prose column (中文使用说明) ──────────────────────
# Row-by-row saving
import time

PROSE_TRANSLATED_COL = '中文使用说明'
print(f'Translating prose column: {PROSE_TRANSLATED_COL}')
print(f'Rows with Chinese: {df[PROSE_TRANSLATED_COL].apply(contains_chinese).sum()}')

for idx in tqdm(range(len(df)), desc='Usage Instructions'):
    val = df.loc[idx, PROSE_TRANSLATED_COL]
    if contains_chinese(val):
        df.loc[idx, PROSE_TRANSLATED_COL] = translate_text(
            val, column_hint='Usage Instructions'
        )
        # Save every 10 rows — if Colab disconnects, reload this file and continue
        if idx % 10 == 0:
            df.to_excel('checkpoint_prose_in_progress.xlsx', index=False)

# Final check
remaining = df[PROSE_TRANSLATED_COL].apply(contains_chinese).sum()
print(f'Rows still containing Chinese: {remaining}')

# Save final checkpoint
df.to_excel('checkpoint_prose_done.xlsx', index=False)
print('Saved → checkpoint_prose_done.xlsx')

Translating prose column: 中文使用说明
Rows with Chinese: 114


Usage Instructions: 100%|██████████| 114/114 [17:49<00:00,  9.38s/it]

Rows still containing Chinese: 0
Saved → checkpoint_prose_done.xlsx


In [ ]:
# ── CELL 17 — Handle any remaining Chinese manually ──────────────────────────
remaining_rows = df[df[PROSE_TRANSLATED_COL].apply(contains_chinese)]
if len(remaining_rows):
    print(f'Manual fix needed for {len(remaining_rows)} rows:')
    for idx, row in remaining_rows.iterrows():
        print(f'\n  Row {idx} — {row.get("产品名称", "")}')
        print(f'  Excerpt: {str(row[PROSE_TRANSLATED_COL])[:200]}')
        # Re-try once with a longer limit
        df.loc[idx, PROSE_TRANSLATED_COL] = translate_text(
            row[PROSE_TRANSLATED_COL],
            column_hint='Usage Instructions'
        )
else:
    print('All rows translated ✓')

All rows translated ✓


In [ ]:
# ── CELL 18 — Translate remaining columns ────────────────────────────────────
OTHER_COLS = {
    '数据来源':       'Data Source',
    '使用方法(兑水)': 'Dilution Method',
}
for cn_col, hint in OTHER_COLS.items():
    if cn_col not in df.columns:
        continue
    print(f'Translating: {cn_col}')
    tqdm.pandas(desc=f'  {hint}')
    df[cn_col] = df[cn_col].progress_apply(
        lambda x, h=hint: translate_text(x, column_hint=h)  # h=hint fixes closure bug
    )

print('\nDone.')

Translating: 数据来源


  Data Source: 100%|██████████| 114/114 [01:07<00:00,  1.68it/s]


Translating: 使用方法(兑水)


  Dilution Method: 100%|██████████| 114/114 [01:20<00:00,  1.42it/s]


Done.


In [ ]:
# Check what the dilution column looks like NOW after Cell 18
print(df['使用方法(兑水)'].value_counts().head(15))

使用方法(兑水)
1 gram diluted with 1 jin of water                                               40
30g diluted with 30 jin of water                                                  7
10 grams diluted in 30 jin of water                                               6
No dilution required                                                              6
Dilute 5000-7000 times into the prepared solution, stir well before spraying.     5
100 grams diluted with 30 jin of water                                            5
Dilute with water according to the instructions and spray                         5
Dilute 5000-7000 times before adding the solution                                 3
50 grams diluted with 30 jin of water                                             2
50g diluted with 30 jin of water                                                  2
Apply directly without dilution                                                   2
200 grams diluted with 30 jin of water                             

In [ ]:
# ── CELL 19 fix — targeted replacement for the 40 affected products ──────────

# Direct fix for the known bad value
df['使用方法(兑水)'] = df['使用方法(兑水)'].str.replace(
    '1 g per 0 L water',
    '1 g per 0.5 L water',
    regex=False
)

# Verify
print(df['使用方法(兑水)'].value_counts().head(5))
jin_remaining = df['使用方法(兑水)'].astype(str).str.contains('jin', case=False).sum()
zero_remaining = df['使用方法(兑水)'].astype(str).str.contains('per 0 L', case=False).sum()
print(f'\nRows still containing "jin": {jin_remaining}')
print(f'Rows containing "per 0 L": {zero_remaining}')

使用方法(兑水)
1 g per 0.5 L water                                                              40
30 g per 15 L water                                                               7
No dilution required                                                              6
10 g per 15 L water                                                               6
Dilute 5000-7000 times into the prepared solution, stir well before spraying.     5
Name: count, dtype: int64

Rows still containing "jin": 0
Rows containing "per 0 L": 0


In [ ]:
# ── CELL 20 — Rename columns to English ──────────────────────────────────────
HEADER_MAP = {
    '产品ID':        'Product ID',
    '产品名称':       'Product Name',
    '英文名称':       'English Name',
    '产品类型':       'Product Type',
    '中文使用说明':   'Usage Instructions',
    '使用方法(兑水)': 'Dilution Method',
    '对应农作物/植物': 'Target Crops',
    '规格':          'Specification',
    '数据来源':       'Data Source',
    '主要成分':       'Main Ingredients',
    '使用方法和剂量':  'Usage Method and Dosage',
}
df.rename(columns={k: v for k, v in HEADER_MAP.items() if k in df.columns}, inplace=True)
print('Columns after rename:')
for col in df.columns:
    print(f'  {col}')

Columns after rename:
  Product ID
  Product Name
  English Name
  Product Type
  Usage Instructions
  Dilution Method
  Target Crops
  Specification
  Data Source
  Main Ingredients
  Usage Method and Dosage
  ingredient_source


In [ ]:
# ── CELL 21 — Final verification ─────────────────────────────────────────────
print('=== FINAL TRANSLATION VERIFICATION ===')
print(f'Shape: {df.shape}')
print(f'\nChinese remaining per column:')
for col in df.columns:
    cn_count = df[col].apply(
        lambda x: contains_chinese(str(x)) if not is_blank(x) else False
    ).sum()
    if cn_count > 0:
        print(f'  WARNING {col}: {cn_count} rows still contain Chinese')
    else:
        print(f'  OK      {col}')

print(f'\nBlank counts after translation:')
print(df.isnull().sum()[df.isnull().sum() > 0])


=== FINAL TRANSLATION VERIFICATION ===
Shape: (114, 12)

Chinese remaining per column:
  OK      Product ID
  OK      Product Name
  OK      English Name
  OK      Product Type
  OK      Usage Instructions
  OK      Dilution Method
  OK      Target Crops
  OK      Specification
  OK      Data Source
  OK      Main Ingredients
  OK      Usage Method and Dosage
  OK      ingredient_source

Blank counts after translation:
Main Ingredients             6
Usage Method and Dosage     17
ingredient_source          108
dtype: int64


In [ ]:
# Spot-check the [DISEASE]/[PEST]/[SYMPTOM] tagging worked
if 'Usage Method and Dosage' in df.columns:
    tagged = df['Usage Method and Dosage'].astype(str).str.contains(
        r'\[DISEASE\]|\[PEST\]|\[SYMPTOM\]'
    ).sum()
    print(f'\n[DISEASE]/[PEST]/[SYMPTOM] tags found in {tagged} rows')


[DISEASE]/[PEST]/[SYMPTOM] tags found in 22 rows


In [ ]:
# Check tagging across both columns
for col in ['Usage Instructions', 'Usage Method and Dosage']:
    tagged = df[col].astype(str).str.contains(
        r'\[DISEASE\]|\[PEST\]|\[SYMPTOM\]', regex=True
    ).sum()
    print(f'{col}: {tagged} rows with tags')

# Show a sample from Usage Instructions to confirm tags look correct
print('\n=== SAMPLE TAGGED ROW — Usage Instructions ===')
sample = df[df['Usage Instructions'].astype(str).str.contains(
    r'\[DISEASE\]|\[PEST\]|\[SYMPTOM\]', regex=True
)].iloc[0]
print(f"Product: {sample['Product Name']}")
print(f"Usage Instructions excerpt:\n{str(sample['Usage Instructions'])[:400]}")

Usage Instructions: 81 rows with tags
Usage Method and Dosage: 22 rows with tags

=== SAMPLE TAGGED ROW — Usage Instructions ===
Product: Citrus Junliqing
Usage Instructions excerpt:
Core control targets (diseases throughout the entire growth period of citrus)
1. Fungal diseases (core): root rot, [DISEASE] downy mildew, [DISEASE] gray mold, [DISEASE] purple spot, [DISEASE] anthracnose, [DISEASE] leaf spot, [DISEASE] wilt, damping-off (high incidence during seedling stage);
2. Bacterial diseases (auxiliary): soft rot, ginger wilt (through microbial colonization + antibacterial 


In [ ]:
# ── CELL 22 — Save final translated file ─────────────────────────────────────
OUTPUT_FILE = 'ProductCatalog_TRANSLATED_v2.xlsx'
df.to_excel(OUTPUT_FILE, index=False)
print(f'Saved → {OUTPUT_FILE}')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

from google.colab import files
files.download(OUTPUT_FILE)

print("""
=== PHASE 1 COMPLETE ✓ ===
Input:  产品目录_CLEANED.xlsx
Output: ProductCatalog_TRANSLATED_v2.xlsx
114 products | 0 Chinese remaining | 6 blanks preserved
81 rows with disease/pest/symptom tags in Usage Instructions
Next: notebook_02_entity_extraction.ipynb
""")

Saved → ProductCatalog_TRANSLATED_v2.xlsx
Shape: (114, 12)
Columns: ['Product ID', 'Product Name', 'English Name', 'Product Type', 'Usage Instructions', 'Dilution Method', 'Target Crops', 'Specification', 'Data Source', 'Main Ingredients', 'Usage Method and Dosage', 'ingredient_source']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== PHASE 1 COMPLETE ✓ ===
Input:  产品目录_CLEANED.xlsx
Output: ProductCatalog_TRANSLATED_v2.xlsx
114 products | 0 Chinese remaining | 6 blanks preserved
81 rows with disease/pest/symptom tags in Usage Instructions
Next: notebook_02_entity_extraction.ipynb



In [52]:
df_raw_names = pd.read_excel('产品目录_CLEANED.xlsx')[['产品ID', '产品名称']]
# At this point 产品名称 is already translated — we need pre-translation names
# Load from the original raw file instead
from google.colab import files
print('Upload 产品目录_ProductCatalog_Mar_6.xlsx to recover Chinese names')
uploaded = files.upload()

# Get the actual file name from the uploaded dictionary
cn_filename = list(uploaded.keys())[0]

df_cn = pd.read_excel(cn_filename,
                       sheet_name='产品目录')[['产品ID', '产品名称']]
df_cn.columns = ['Product ID', 'product_name_cn']

df = df.merge(df_cn, on='Product ID', how='left')
print(f'Shape after merge: {df.shape}')
print('\nSpot check:')
for pid in ['AF0001', 'PN0001', 'AF0035']:
    row = df[df['Product ID'] == pid].iloc[0]
    print(f"  {pid} | CN: {row['product_name_cn']} | EN: {row['Product Name']}")

# Re-save the final file with CN names included
df.to_excel('ProductCatalog_TRANSLATED_v2.xlsx', index=False)
files.download('ProductCatalog_TRANSLATED_v2.xlsx')
print('\nSaved → ProductCatalog_TRANSLATED_v2.xlsx (now includes product_name_cn)')

Upload 产品目录_ProductCatalog_Mar_6.xlsx to recover Chinese names


Saving 产品目录_ProductCatalog_Mar.6.xlsx to 产品目录_ProductCatalog_Mar.6 (1).xlsx
Shape after merge: (114, 13)

Spot check:
  AF0001 | CN: 柑橘菌立清 | EN: Citrus Junliqing
  PN0001 | CN: 苏云金杆菌4000IU/微升 | EN: Bacillus thuringiensis 4000 IU/μL
  AF0035 | CN: 百菌灵 | EN: Carbendazim


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved → ProductCatalog_TRANSLATED_v2.xlsx (now includes product_name_cn)
